build from scratch a causal attention


In [ ]:


    import torch 
    import torch.nn as nn 
    import math  # Nécessaire pour math.sqrt
    import torch.functional as F
    #definition de l'attention(dropout, context lenght,d_in, d_out,dropout )

    class causalAttention(nn.Module):
        def __init__(self,context_length, d_in, d_out,qkv_bias = False, dropout= 0.4):
            super().__init__()
            self.d_out = d_out

            #definisionns des matrices qkv
            self.wq = nn.Linear(d_in, d_out,bias = qkv_bias)
            self.wk = nn.Linear(d_in, d_out, bias = qkv_bias)
            self.wv = nn.Linear(d_in, d_out, bias = qkv_bias)
            self.dropout = nn.Dropout(dropout)

            self.register_buffer(
                'mask',
                torch.triu(torch.ones(context_length, context_length),diagonal = 1
                )
            )          

        

        def forward(self, x):
            #initialisation de la dim du model,qu'on recupere pour activer notre sofmax 
            #~initialisation de la shape de X
            b ,num_tokens, d_in = x.shape
            #Création des vexteur a partir de l'entree de l'input x @ les matrices  x = 1, 6, 512 @ matrices (512,512) = 6,512
            Q = self.wq(x)# (b, num_tokens, d_out
            K = self.wk(x)# (b, num_tokens, d_out
            V = self.wv(x)# (b, num_tokens, d_out

            #on passe de Q = (6,512), k = (6,512) =>  Q = (6,512), k = (512,6)  pour pouvoir faire nos attention scores donc b nums token par nums token 
            attn_scores = Q @ K.transpose(1,2)

            #applixation du masking sur la matrices 6,6, a l'inférence, C'est sequentiel, donc on a (1,) physiquement le prochain token 
            # n'est pas là donc masking inutile, mais on le garde a but pédagogique
            # Masquage des mots futurs
            mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
            attn_scores.masked_fill_(mask_bool, -float('inf'))
            #application du sofmatx aux logits 
            attn_weights = torch.softmax(attn_scores / math.sqrt(self.d_out), dim = -1)
            attn_dropout = self.dropout(attn_weights)
            contextVector  = attn_dropout @ V 
            return contextVector
    #definition du mlp(2 hidden layer, oout proj, d_in , d_out, dropout)

    class mlp(nn.Module): 
        def __init__(self, d_in, d_out,hidden1, hidden2,dropout = 0.3): 
            super().__init__()

            self.l1 = nn.Linear(d_in, hidden1) #Calcul d'un vecteur de x dimension @ le nombre de neurones eg = x1w1..., x512w512  
            self.l2 = nn.Linear(hidden1, hidden2)
            self.out = nn.Linear(hidden2, d_out)
            self.dropout = nn.Dropout(dropout)


        def forward(self, x):
            x = self.dropout(self.l2(F.relu(self.l1(x))))
            x = self.out(x)
            return x 
    #definition du block transformer(context length, d_model,dropout)

    class TransformerBlock(nn.Module):
        def __init__(self, context_length, d_model, dropout):
            super().__init__()

            self.ln1 = nn.LayerNorm(d_model)
            self.ln2 = nn.LayerNorm(d_model)
    
            self.attention = causalAttention(
                context_length = context_length,
                d_in = d_model,
                d_out = d_model,
                dropout = dropout
            )

            self.mlp = mlp(
            d_in=d_model,
            d_out=d_model,
            hidden1=4 * d_model,
            hidden2=4 * d_model,
            dropout=dropout
            )
            
        def forward(self,x ):
            x = x + self.attention(self.ln1(x))
            x = x + self.mlp(self.ln2(x))
            return x 


    batch_size = 2
    context_length = 2048
    d_in = 512
    d_out = 512
    dropout_rate = 0.4

    # 2. Création de l'entrée tensor de dimension (batch, sequence_length, d_in)
    # Ici: batch=2, sequence_length=6 tokens, dimension=512
    x = torch.randn(batch_size, 6, d_in)
    print("1. Entrée x :", x.shape)

    # 3. Initialisation du modèle avec ses arguments


    # 4. Appel du forward pass
    model = TransformerBlock(d_model=d_in, context_length=context_length, dropout=dropout_rate)
    predict = model(x)
    print("2. Sortie (context_vec) :", predict.shape)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (739820393.py, line 17)